In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
# set folders
try:
    onedrive_folder = os.environ['OneDriveCommercial'].replace('\\','/') + '/'
except:
    onedrive_folder = os.environ['OneDrive'].replace('\\','/') + '/'

data_folder = 'Documents/projects/frailty/ML analysis/data/'
results_folder = 'Documents/projects/frailty/ML analysis/results/'

# set files
in_file = '2025.04-evelyn-15.csv'
out_file = 'cleaned_data_2026-08-05_unktags_2014.csv'
today = datetime.now().strftime('%Y%m%d_%H%M%S')
table1_file = f'table1_{today}.xlsx'

In [ ]:
# read in the data
slsr = pd.read_csv(onedrive_folder+data_folder+in_file, low_memory=True)
slsr.info()

In [ ]:
# drop final_lsoa, efi_total, efi_cat, strk_d, strk_m
slsr2 = slsr.drop(['final_lsoa','efi_total','efi_cat','strk_d','strk_m'],axis=1)
slsr2.info()

In [ ]:
# subset data to use only everything from 2014 onwards because the cancer and kidney variables only started being collected from 2014 onwards
slsr3 = slsr2[slsr2['strk_y'] >= 2014]
slsr3.describe(include='all')

In [ ]:
# combine variables
slsr4 = slsr3.copy()

# define function to check variables
def check_variable(row, variable_cols):
    values = row[variable_cols]
    
    # Case 1: All values are missing
    if values.isna().all():
        return pd.NA
    
    # Case 2: At least one value is 2.0 (regardless of other missing values)
    if (values == 2.0).any():
        return 1
    
    # Case 3: No value is 2.0, but we have at least one non-missing value
    return 0

# create new cancer variable
cancer_cols = ['st_rec','mst_rec','leuk_rec','mlymp_rec']
# apply function to cancer variables
slsr4['cancer'] = slsr4.apply(check_variable, axis=1, variable_cols=cancer_cols) # no need to use lambda, just directly pass kwargs after the axis=1 argument
slsr4['cancer'] = slsr4['cancer'].astype('Int64')

# create new variable - yes/no for kidney issues
kidney_cols = ['kidney_rec','rrt_rec']
# apply function to cancer variables
slsr4['kidney'] = slsr4.apply(check_variable, axis=1, variable_cols=kidney_cols)
slsr4['kidney'] = slsr4['kidney'].astype('Int64')

# inspect
slsr4.info()

In [ ]:
slsr5 = slsr4.copy()

# change strk_dt and use_dod from obj to datetime format
slsr5['strk_dt'] = pd.to_datetime(slsr5['strk_dt'])
slsr5['use_dod'] = pd.to_datetime(slsr5['use_dod'])

# make arrival at A&E into usable datetime format
slsr5['ae_dt'] = pd.to_datetime({'day': slsr5['ae_d'], 'month': slsr5['ae_m'], 'year': slsr5['ae_y']}, errors='coerce') # make arrival date into date format
slsr5['arr_time'] = slsr5['arr_time'].fillna(0).astype(int).astype(str).str.zfill(4) 
# have to fill NA's with zeros otherwise we can't generate a valid datetime for the patient when date is available but time is missing
# have to convert from float to int otherwise we get things like 800.0, and when you convert to str, there's already 4 digits and zfill doesn't bother to add a zero in front
# pad arrival time with leading zeros where applicable to achieve a strictly 4-digit column
slsr5['arr_time'] = slsr5['arr_time'].str[:2] + ':' + slsr5['arr_time'].str[2:] # put colons into arrival time to get this, e.g. 09:00
slsr5['arr_datetime'] = pd.to_datetime(slsr5['ae_dt'].astype(str) + ' ' + slsr5['arr_time'], errors = 'coerce') # convert to datetime. insert a space btwn date and time, it's a separator to make clear where the date ends and the time begins.

# make stroke date into usable datetime format
slsr5['ctime'] = slsr5['ctime'].fillna(0).astype(int).astype(str).str.zfill(4)
# if ctime=2400, change this to 2359, otherwise it becomes NA when you join it with date
slsr5['ctime'] = np.where(slsr5['ctime']=='2400', '2359', slsr5['ctime'])
# if ctime=9999, change this to 0000, otherwise it becomes NA when you join it with date
slsr5['ctime'] = np.where(slsr5['ctime']=='9999', '0000', slsr5['ctime'])
slsr5['ctime'] = slsr5['ctime'].str[:2] + ':' + slsr5['ctime'].str[2:]
slsr5['strk_datetime'] = pd.to_datetime(slsr5['strk_dt'].astype(str) + ' ' + slsr5['ctime'], errors = 'coerce')

# find the difference (in hours) between stroke time and arrival time
slsr5['onset_to_presentation'] = (slsr5['arr_datetime'] - slsr5['strk_datetime']).dt.total_seconds() / 3600
# slsr3['arr_datetime'] - slsr3['strk_datetime']: gives you a timedelta object
# Example output:
# 0   0 days 02:15:00
# 1   0 days 01:30:00
# 2   0 days 05:45:00
# dtype: timedelta64[ns]
# extract useful format from it, in seconds. then divide to get hours

# inspect
slsr5.info()

In [ ]:
# remove variables that aren't needed
cols_to_drop = cancer_cols + kidney_cols
slsr6 = slsr5.drop(cols_to_drop, axis=1)
slsr6.info()

In [ ]:
# describe the df
slsr6.describe(include='all')
# include=all will include categorical variables

In [ ]:
print(f'Before removing extreme ages: {slsr6.shape[0]} participants')

# remove people with extreme ages
slsr7 = slsr6[(slsr6['age'] >= 18) & (slsr6['age'] <= 120)]
print(f'After removing extreme ages: {slsr7.shape[0]} participants remaining')

# remove people whose onset_to_presentation time is negative or do not have onset_to_presentation time
slsr7 = slsr7[(slsr7['onset_to_presentation'] >= 0) | (slsr7['onset_to_presentation'].isna())]
print(f'After removing people with negative or missing onset to presentation times: {slsr7.shape[0]} participants remaining')

slsr7.info()

In [ ]:
# look at the categories present in each variable
for col in slsr7.columns:
    print(slsr7[col].value_counts(dropna=False))

In [ ]:
# convert all binary variables (1s/2s and yes/no and T/F) with unknowns to 0s/1s to ease input into ML libraries

slsr8 = slsr7.copy()

# sex              8684 non-null   float64  1=Male 2=Female
# rfpdep           6720 non-null   float64  1-no, 2-yes
# rfpihd           3885 non-null   float64  1-no, 2-yes
# bi0.25           4362 non-null   float64 1=">=15"  2="<15"
# bi1              4184 non-null   float64 1=">=15"  2="<15"
cols_bi = ['sex','rfpdep','rfpihd','bi0.25','bi1']
for col in cols_bi: # change from 1s/2s to 0s/1s
    slsr8[col] = slsr8[col].replace({1.0: 0, 2.0: 1}) # 0 for male, 1 for female

# NA's present, weird category 3.0 that is not in codebook (we will assume these are unknowns)
# hospital         8548 non-null   float64 1=No, 2=Yes
# adm_ae           4159 non-null   float64 8548 non-null   float64 1=No, 2=Yes
# adm_hos          8595 non-null   float64 1=No, 2=Yes
# incont           8310 non-null   float64 1=No, 2=Yes
# stroke           8647 non-null   float64 1=infarct 2=haemorrhage
# rfdrik           8108 non-null   float64 1=No, 2=Yes, NA's present
# rfpmi            8414 non-null   float64 1=No, 2=Yes, NA's present
# rfpaf            8395 non-null   float64 1=No, 2=Yes, NA's present
# rfphyp           8396 non-null   float64 1=No, 2=Yes, NA's present
# rfpdia           8623 non-null   float64 1=No, 2=Yes, NA's present
# rfptia           8387 non-null   float64 1=No, 2=Yes, NA's present
# rfpcho           7259 non-null   float64 1=No, 2=Yes, NA's present
# rfppvd           8266 non-null   float64 1=No, 2=Yes, NA's present
# rfpccf           7185 non-null   float64 1=No, 2=Yes, NA's present
# rfpang           5689 non-null   float64 1=No, 2=Yes, NA's present
cols_tri = ['hospital','adm_ae','adm_hos','incont','stroke','rfdrik','rfpmi','rfpaf','rfphyp','rfpdia','rfptia','rfpcho','rfppvd','rfpccf','rfpang']
for col in cols_tri:
    slsr8[col] = slsr8[col].map({1.0: 0, 2.0: 1}) # map() converts everything not indicated here into pd.NA by default
    slsr8[col] = pd.to_numeric(slsr8[col], errors = 'coerce') # map converts things to objects. we have to convert it back to numeric after, and coerce everything not 0/1 into pd.NA

# NA's present, weird categories 0.0 and 3.0 that are not in codebook (we will assume these are unknowns)
# dement_rec       3245 non-null   float64 1=No, 2=Yes, NA's present
# copd_rec         3247 non-null   float64 1=No, 2=Yes, NA's present
cols_quad = ['dement_rec','copd_rec']
for col in cols_quad:
    slsr8[col] = slsr8[col].map({0.0: pd.NA, 1.0: 0, 2.0: 1, 3.0: pd.NA})
    slsr8[col] = pd.to_numeric(slsr8[col], errors='coerce')

# final_antihyp_p  8348 non-null   object   2 levels no/yes. change to 0/1     
# final_antidia_p  8466 non-null   object   2 levels no/yes. change to 0/1     
# final_anticho_p  7967 non-null   object   2 levels no/yes. change to 0/1     
# final_ac_p       8365 non-null   object   2 levels no/yes. change to 0/1     
# final_ap_p       8365 non-null   object   2 levels no/yes. change to 0/1
med_cols = ['final_antihyp_p','final_antidia_p','final_anticho_p','final_ac_p','final_ap_p']
for col in med_cols:
    slsr8[col] = slsr8[col].replace({'No':0, 'Yes':1})
    slsr8[col] = pd.to_numeric(slsr8[col], errors='coerce') # coerce: all NA's remain NA's while converting to float

# change these categorical variables into objects with proper levels, to facilitate one hot encoding later
#  eth6cat          8822 non-null   int64    1=White 2=Black Carribean 3=Black African 4=Black Other 5=Other 99= Unknown/not stated     
conditions = [
    slsr8['eth6cat']==1.0,
    slsr8['eth6cat']==2.0,
    slsr8['eth6cat']==3.0,
    slsr8['eth6cat']==4.0,
    slsr8['eth6cat']==5.0,
    slsr8['eth6cat']==99.0
]

choices = [
    'White',
    'Black Carribean',
    'Black African',
    'Black Other',
    'Other',
    pd.NA
]

slsr8['eth6cat'] = np.select(conditions, choices, default=pd.NA)

#  rfsmok           8266 non-null   float64  1=Never 2=Ex-smoker 3=Current 4=unk
conditions = [
    slsr8['rfsmok']==1.0,
    slsr8['rfsmok']==2.0,
    slsr8['rfsmok']==3.0,
    slsr8['rfsmok']==4.0,
    slsr8['rfsmok'].isna()
]

choices = [
    'Never',
    'Ex-smoker',
    'Current smoker',
    pd.NA,
    pd.NA
]

slsr8['rfsmok'] = np.select(conditions, choices, default=pd.NA)

# livcon           7801 non-null   float64  1=Private household alone 2=Private household with others 3=Sheltered home 4=Residential home 5=Nursing home 6=Community hospital 7=Private hospital 8=Other 10=Long term hospital care 11=Care home
conditions = [
    slsr8['livcon']==1.0,
    slsr8['livcon']==2.0,
    slsr8['livcon']==3.0,
    slsr8['livcon']==4.0,
    slsr8['livcon']==5.0,
    slsr8['livcon']==6.0,
    slsr8['livcon']==7.0,
    slsr8['livcon']==8.0,
    slsr8['livcon']==10.0,
    slsr8['livcon']==11.0,
    slsr8['livcon'].isna()
]

choices = [
    'Private household alone',
    'Private household with others',
    'Sheltered home',
    'Residential home',
    'Nursing home',
    'Community hospital',
    'Private hospital',
    'Other',
    'Long term hospital care',
    'Care home',
    pd.NA
]

slsr8['livcon'] = np.select(conditions, choices, default=pd.NA)

# # ddestin for multinomial logistic regression with 4 categories
# # ddestin           2725 non-null   float64  1=private household alone 2=private household with carer 3=sheltered housing 4=residential home 5=nursing home 6=community hospital 7=private hospital 8=death 9=other 10=long term hospital care 11=care home
# conditions = [
#     slsr8['ddestin'] == 1.0,
#     slsr8['ddestin'].isin([2.0, 3.0]),
#     slsr8['ddestin'].isin([4.0, 5.0, 11.0, 6.0, 7.0, 10.0]),  # all institution
#     (slsr8['ddestin'].isin([88.0, 888.0, 99.0, 9.0])) | (slsr8['ddestin'].isna()),
#     slsr8['ddestin'] == 8.0
# ]

# choices = [
#     'Home_independent',
#     'Home_support',
#     'Institutional_care',
#     'Unclear',
#     pd.NA  # deaths
# ]

# slsr8['ddestin_ordinal'] = np.select(conditions, choices, default=pd.NA)

# ddestin for binary logistic regression with 2 categories
# ddestin           2725 non-null   float64  1=private household alone 2=private household with carer 3=sheltered housing 4=residential home 5=nursing home 6=community hospital 7=private hospital 8=death 9=other 10=long term hospital care 11=care home
conditions = [
    slsr8['ddestin'].isin([1.0, 2.0, 3.0]),
    slsr8['ddestin'].isin([4.0, 5.0, 11.0, 6.0, 7.0, 10.0])
]

choices = [
    0, 
    1
]  # 0=Home, 1=Institution

slsr8['ddestin_binary'] = np.select(conditions, choices, default=pd.NA)
slsr8['ddestin_binary'] = slsr8['ddestin_binary'].astype('Int64') # change to Int64 as it can take NA's
slsr8['ddestin_binary'] = slsr8['ddestin_binary'].astype('float64') # change again to float otherwise it will throw an error during the onehotencoding because the OHE cannot parse NA's in a Int64 object (pd.NA). it can only parse NA's in a float64 object (np.nan)

# we give missing tags for certain variables where missingness could possibly be informative
# glas_cs          8208 non-null   float64 
slsr8['glas_cs_missing'] = slsr8['glas_cs'].isnull().astype(int)
slsr8['glas_cs'] = slsr8['glas_cs'].fillna(slsr8['glas_cs'].median())

#  mtot             2880 non-null   float64
slsr8['mtot_missing'] = slsr8['mtot'].isnull().astype(int)
slsr8['mtot'] = slsr8['mtot'].fillna(slsr8['mtot'].median())

# swa_test         8113 non-null   float64 1=Pass 2=Fail
slsr8['swa_test'] = slsr8['swa_test'].map({1.0: 'Pass', 2.0: 'Fail'})
slsr8['swa_test'] = slsr8['swa_test'].fillna('Missing')

# cogcat           4236 non-null   float64  1=cognitively intact (8-10 AMT and 24+ MMSE) 2=cognitively impaired (0-7 AMT and <24 MMSE)
slsr8['cogcat'] = slsr8['cogcat'].map({1.0: 'cognitively intact', 2.0: 'cognitively impaired'})
slsr8['cogcat'] = slsr8['cogcat'].fillna('Missing')

#  nihss            5531 non-null   float64 1=mild(<=4) 2=moderate(5-20) 3=severe(>20)
slsr8['nihss'] = slsr8['nihss'].map({1.0: 'mild', 2.0: 'moderate', 3.0: 'severe'})   
slsr8['nihss'] = slsr8['nihss'].fillna('Missing')   

# mtaet6           6416 non-null   float64 1=LAA 2=CE 3=SVO 4=OTH 5=UND 6=CONC 77=PICH 88=SAH
conditions = [
    slsr8['mtaet6']==1.0,
    slsr8['mtaet6']==2.0,
    slsr8['mtaet6']==3.0,
    slsr8['mtaet6']==4.0,
    slsr8['mtaet6']==5.0,
    slsr8['mtaet6']==6.0,
    slsr8['mtaet6']==77.0,
    slsr8['mtaet6']==88.0,
    slsr8['mtaet6'].isna()
]

choices = [
    'LAA',
    'CE',
    'SVO',
    'OTH',
    'Undetermined',
    'Concommitant',
    'PICH',
    'SAH',
    'Missing'
]

slsr8['mtaet6'] = np.select(conditions, choices, default=pd.NA)

# subtype 1=TACI 2=PACI 3=POCI 4=LACI 5=Infarct unspecified 6=PICH 7=SAH 8=Unclassified 9=Unknown
conditions = [
    slsr8['subtype']==1.0,
    slsr8['subtype']==2.0,
    slsr8['subtype']==3.0,
    slsr8['subtype']==4.0,
    slsr8['subtype']==5.0,
    slsr8['subtype']==6.0,
    slsr8['subtype']==7.0,
    slsr8['subtype']==8.0,
    slsr8['subtype']==9.0
]

choices = [
    'TACI',
    'PACI',
    'POCI',
    'LACI',
    'Infarct unspecified',
    'PICH',
    'SAH',
    'Unclassified',
    'Unknown'
]

# Code 9 ("Unknown") is almost certainly informative missingness:
    # The patient didn't get the full diagnostic workup (echocardiogram, cardiac monitoring, vessel imaging)
    # Reasons: early death, rapid discharge, palliative care, or transfer to another facility
    # These reasons correlate with outcomes

slsr8['subtype'] = np.select(conditions, choices, default=pd.NA)

#  45  acacort          176 non-null    object 
#  46  acmcort          1627 non-null   object 
#  47  acmbasal         1235 non-null   object 
#  48  acpcort          360 non-null    object 
#  49  acpthal          227 non-null    object 
#  50  acpbstem         326 non-null    object 
#  51  acpcereb         338 non-null    object 
# change all NA's to 'none' because it means that structure in the brain was not affected at all. not just missing.
brain_loc_vars = ['acacort','acmcort','acmbasal','acpcort','acpthal','acpbstem','acpcereb']
slsr8[brain_loc_vars] = slsr8[brain_loc_vars].fillna('None')

In [ ]:
slsr8.info()

In [ ]:
# create the mortality 3m and 1y outcomes (0/1)
# this has to be done before the onehot encoding, because we need to drop the datetime cols before feeding into the onehotencoder. NumPy does not properly support pandas datetime dtype inside mixed arrays. So one of two things happens:
    # Case 1 — It becomes object. Your entire matrix may become dtype object.
    # Case 2️ — It gets converted to integer timestamps. Datetime becomes its internal integer representation: number of nanoseconds since 1970-01-01. Example: 2020-01-01 becomes something like: 1577836800000000000
# Most ML models:
    # Cannot handle datetime dtype
    # Should not receive raw timestamps
    # Need engineered numeric features
    # Raw datetime has no natural meaning to a model.

slsr8['death0.25'] = np.where(
    (slsr8['use_dod'].notna()) & 
    (slsr8['use_dod'] <= slsr8['strk_dt'] + pd.Timedelta(days=90)),
    1, 0
)

slsr8['death1'] = np.where(
    (slsr8['use_dod'].notna()) & 
    (slsr8['use_dod'] <= slsr8['strk_dt'] + pd.Timedelta(days=365)),
    1, 0
)

# drop the strk_dt and death_dt cols
slsr9 = slsr8.drop(['strk_dt','use_dod'], axis=1)

slsr9.info()

In [ ]:
# remove unnecessary vars
slsr9.drop(columns=['mtaet6','cogcat','mtot','ctime','arr_time','ae_m','ae_d','ae_y','ae_dt','arr_datetime','strk_datetime','mtot_missing','ddestin'], inplace=True, axis=1)
slsr9.info()

#### CCA
##### 1. remove variables with lots of NA's
##### 2. drop all observations with at least 1 NA (CCA)

In [ ]:
# remove variables with lots of NA's
# adm_ae, cogcat, mtot, rfpihd
# rfpihd was only collected from 1995-2007. we will only include data from 2014 onwards so this will be completely NA for our period of interest
slsr10 = slsr9.drop(columns = ['rfpihd','rfpang'], axis=1)
slsr10.info()

In [ ]:
# do a CCA
# drop rows with NA's in 1 or more of these variables: 
cols_not_to_touch = ['bi0.25','bi1','death0.25','death1','efi_score','ddestin_binary']
cols_cca = [col for col in slsr10.columns if col not in cols_not_to_touch]
slsr11 = slsr10.dropna(subset=cols_cca)
slsr11.info()

In [ ]:
len(slsr11[slsr11['strk_y'] <= 2022])

In [ ]:
# then do one-hot encoding on all categorical variables
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Define which columns are categorical
categorical_cols = slsr11.select_dtypes(include=['object']).columns
numerical_cols = slsr11.select_dtypes(include=['int64', 'int32', 'float64', 'Int64']).columns
# Let sklearn detect categorical columns properly instead of checking dtype == 'object'

# Create preprocessor 
# ColumnTransformer is like a pipeline for columns - it applies different transformations to different columns.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),  
        # 'num': Name of this transformer (you could call it anything, e.g., 'numeric')
        # 'passthrough': Special string that means "don't transform, just pass the data through as-is"
        # numerical_cols: List like ['age', 'income'] - these columns will be kept unchanged
        ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ],
        # 'cat': Name of this transformer
        # OneHotEncoder(...): The actual transformer object that will one-hot encode the data
        # categorical_cols: List like ['cancer_type', 'smoking_status'] - these will be one-hot encoded
        # sparse_output=False: Returns a normal numpy array (easier to work with). If True, returns a sparse matrix (saves memory but harder to use)
        # handle_unknown='ignore': If your test data has a category not seen in training, it will create all zeros instead of throwing an error
    # transformers=[...] is a list of tuples, each with 3 elements:
        # Name: 'num' and 'cat' - just labels you choose
        # Transformer: What to do with these columns
        # Columns: Which columns to apply it to
    verbose_feature_names_out=False
    # so that sklearn won't add num__ and cat__ prefixes to the start of every variable.
    )

# Fit and transform
slsr11_encoded = preprocessor.fit_transform(slsr11).astype('float64')
# After fitting, you can access the fitted transformers by their names
# coerce all to float64, otherwise Column Transformer auto changes all dtypes to object when it sees a mix of numerical formats like int64, float64, Int64, int32

# Get feature names
feature_names = preprocessor.get_feature_names_out()
# cat_transformer = preprocessor.named_transformers_['cat']: This gives you the fitted OneHotEncoder object
# .get_feature_names_out(categorical_cols): This generates the names of the one-hot encoded columns in a numpy array. we convert to list after because it's just easier to work with lists but it doesn't affect anything

# Create DataFrame
slsr12 = pd.DataFrame(slsr11_encoded, columns=feature_names)

# inspect
slsr12.info()


In [ ]:
slsr11_encoded.dtype

In [ ]:
slsr12.describe(include='all')

In [ ]:
# write to csv
slsr12.to_csv(onedrive_folder+data_folder+out_file, index=False)

#### MAKE TABLE 1

In [ ]:
# change all the binary variables to factor
change_vars = ['sex','hospital','adm_ae','adm_hos','incont','rfdrik','cancer','kidney'] + [var for var in slsr11.columns if var.startswith('rfp') or var.startswith('final_')] + ['dement_rec','copd_rec']

for var in change_vars:
    print(slsr11[var].value_counts(dropna=False))

In [ ]:
# change all binary 1s/0s to yes/no
slsr12 = slsr11.copy()

for var in change_vars:
    slsr12[var] = np.where(slsr12[var]==0, 'No', 'Yes')

slsr12['stroke'] = np.where(slsr12['stroke']==0, 'Ischemic', 'Hemorrhagic')
slsr12['sex'] = np.where(slsr12['stroke']==0, 'Male', 'Female')

# change stroke year to categorical
conditions = [
    slsr12['strk_y'] < 2017,
    slsr12['strk_y'] < 2020,
    slsr12['strk_y'] < 2023,
    slsr12['strk_y'] >= 2023
]
choices = [
    '2014-2016',
    '2017-2019',
    '2020-2022',
    '2023-2025'
]
slsr12['strk_y_binned'] = np.select(conditions, choices, default=pd.NA)

# drop all the brain location variables
slsr12.drop(columns=[var for var in slsr12.columns if var.startswith('ac')], inplace=True)

# inspect
slsr12.info()

In [ ]:
slsr13 = slsr12.copy()

In [ ]:
from scipy.stats import chi2_contingency, fisher_exact, ttest_ind, mannwhitneyu, normaltest

# Function to calculate p-value for categorical variables
def calc_cat_pvalue(var, dead_group, alive_group):
    """
    Calculate appropriate p-value for categorical variable
    Returns p-value and test used
    """
    # Create contingency table for all categories
    dead_counts = dead_group[var].value_counts(dropna=False)
    alive_counts = alive_group[var].value_counts(dropna=False)
    
    # Align the indices to include all categories
    all_cats = pd.Index(set(dead_counts.index) | set(alive_counts.index))
    dead_counts = dead_counts.reindex(all_cats, fill_value=0)
    alive_counts = alive_counts.reindex(all_cats, fill_value=0)
    
    # Create 2xk contingency table
    contingency = pd.DataFrame([dead_counts, alive_counts])
    
    # Check expected frequencies
    chi2, p_value, dof, expected = chi2_contingency(contingency)
    
    # If any expected cell count < 5, use Fisher's exact for 2x2, or note for larger tables
    if (expected < 5).any():
        if len(all_cats) == 2:  # 2x2 table
            # Fisher's exact test
            table_2x2 = contingency.values
            odds_ratio, p_value = fisher_exact(table_2x2)
            test_used = "Fisher's exact"
        else:
            # For larger tables with low expected counts, keep chi-square but note
            test_used = "Chi-square (low expected counts)"
    else:
        test_used = "Chi-square"
    
    return p_value, test_used

# Function to calculate p-value for numerical variables
def calc_num_pvalue(var, dead_group, alive_group):
    """
    Calculate appropriate p-value for numerical variable
    Returns p-value and test used
    """
    dead_vals = dead_group[var].dropna()
    alive_vals = alive_group[var].dropna()
    
    # Check if both groups have data
    if len(dead_vals) == 0 or len(alive_vals) == 0:
        return np.nan, "Insufficient data"
    
    # Test for normality (simplified - using D'Agostino's test)
    _, p_norm_dead = normaltest(dead_vals) if len(dead_vals) >= 8 else (None, 0.05)
    _, p_norm_alive = normaltest(alive_vals) if len(alive_vals) >= 8 else (None, 0.05)
    
    # If both groups appear normal (p_norm > 0.05) or large samples (>30), use t-test
    if (len(dead_vals) >= 30 and len(alive_vals) >= 30) or (p_norm_dead > 0.05 and p_norm_alive > 0.05):
        # Independent t-test
        t_stat, p_value = ttest_ind(dead_vals, alive_vals, equal_var=False)
        # Welch’s t-test — almost always preferable in observational data. assumes unequal variances
        test_used = "t-test"
    else:
        # Mann-Whitney U test
        u_stat, p_value = mannwhitneyu(dead_vals, alive_vals)
        test_used = "Mann-Whitney U"
    
    return p_value, test_used

In [ ]:
# split into train and test sets so we can generate table 1s
test_df = slsr13[slsr13['strk_y']>2020]
print(len(test_df))

train_df = slsr13[slsr13['strk_y']<=2020]
print(len(train_df))

### Death 3m

In [ ]:
# =====================================
# OVERALL
# =====================================

# table 1
df_death3m_tab1 = slsr13.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
df_death3m_table1_rows = []

cat_vars = [var for var in df_death3m_tab1.columns if df_death3m_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_3m = df_death3m_tab1[df_death3m_tab1['death0.25'] == 1]
alive_3m = df_death3m_tab1[df_death3m_tab1['death0.25'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_3m)
n_alive = len(alive_3m)
n_total = len(df_death3m_tab1)

print(f"Dead at 3m: n={n_dead}, Alive at 3m: n={n_alive}, Total: n={n_total}")

# Loop through all variables except the outcome
for var in df_death3m_tab1.drop(['bi0.25','bi1','death0.25','death1','ddestin_binary','strk_y'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = df_death3m_tab1[var].value_counts(dropna=False, sort=False).index.tolist()
        # by default value_counts() returns counts sorted by frequency (highest first). with sort=False, it will returns counts in category order (as defined in your categorical)

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_3m, alive_3m)
        
        for level in levels:
            # dead group
            count_dead = (dead_3m[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_3m[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            df_death3m_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Dead at 3m': f"{count_dead} ({prop_dead:.1f}%)",
                'Alive at 3m': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_3m, alive_3m)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_3m[var].mean()
        dead_sd = dead_3m[var].std()
        dead_median = dead_3m[var].median()
        dead_iqr = dead_3m[var].quantile(0.75) - dead_3m[var].quantile(0.25)

        # alive group
        alive_mean = alive_3m[var].mean()
        alive_sd = alive_3m[var].std()
        alive_median = alive_3m[var].median()
        alive_iqr = alive_3m[var].quantile(0.75) - alive_3m[var].quantile(0.25)

        # Add row for numerical variable
        df_death3m_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Dead at 3m': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Alive at 3m': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
df_death3m_table1 = pd.DataFrame(df_death3m_table1_rows)

# Add sample sizes to column headers
df_death3m_table1.columns = [
    'Variable', 
    'Categories', 
    f'Dead at 3m (n={n_dead})', 
    f'Alive at 3m (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 3-month Mortality (OVERALL)")
print("="*80)
print(df_death3m_table1.to_string(index=False))

# save to excel
# First sheet (creates file)
df_death3m_table1.to_excel(onedrive_folder+results_folder+table1_file, sheet_name='death_3m_overall', index=False)

In [ ]:
# =====================================
# TRAIN DATA
# =====================================

# table 1
train_df_death3m_tab1 = train_df.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
train_df_death3m_table1_rows = []

cat_vars = [var for var in train_df_death3m_tab1.columns if train_df_death3m_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_3m = train_df_death3m_tab1[train_df_death3m_tab1['death0.25'] == 1]
alive_3m = train_df_death3m_tab1[train_df_death3m_tab1['death0.25'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_3m)
n_alive = len(alive_3m)
n_total = len(train_df_death3m_tab1)

print(f"Dead at 3m: n={n_dead}, Alive at 3m: n={n_alive}")

# Loop through all variables except the outcome
for var in train_df_death3m_tab1.drop(['bi0.25','bi1','death0.25','death1'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = train_df_death3m_tab1[var].value_counts(dropna=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_3m, alive_3m)
        
        for level in levels:
            # dead group
            count_dead = (dead_3m[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_3m[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            train_df_death3m_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Dead at 3m': f"{count_dead} ({prop_dead:.1f}%)",
                'Alive at 3m': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_3m, alive_3m)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_3m[var].mean()
        dead_sd = dead_3m[var].std()
        dead_median = dead_3m[var].median()
        dead_iqr = dead_3m[var].quantile(0.75) - dead_3m[var].quantile(0.25)

        # alive group
        alive_mean = alive_3m[var].mean()
        alive_sd = alive_3m[var].std()
        alive_median = alive_3m[var].median()
        alive_iqr = alive_3m[var].quantile(0.75) - alive_3m[var].quantile(0.25)

        # Add row for numerical variable
        train_df_death3m_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Dead at 3m': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Alive at 3m': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
train_df_death3m_table1 = pd.DataFrame(train_df_death3m_table1_rows)

# Add sample sizes to column headers
train_df_death3m_table1.columns = [
    'Variable', 
    'Categories', 
    f'Dead at 3m (n={n_dead})', 
    f'Alive at 3m (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 3-month Mortality (TRAINING SET)")
print("="*80)
print(train_df_death3m_table1.to_string(index=False))

# save to excel
# First sheet (creates file)
train_df_death3m_table1.to_excel(onedrive_folder+results_folder+table1_file, sheet_name='death_3m_train', index=False)

In [ ]:
# =====================================
# TEST DATA
# =====================================

# table 1
test_df_death3m_tab1 = test_df.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
test_df_death3m_table1_rows = []

cat_vars = [var for var in test_df_death3m_tab1.columns if test_df_death3m_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_3m = test_df_death3m_tab1[test_df_death3m_tab1['death0.25'] == 1]
alive_3m = test_df_death3m_tab1[test_df_death3m_tab1['death0.25'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_3m)
n_alive = len(alive_3m)
n_total = len(test_df_death3m_tab1)

print(f"Dead at 3m: n={n_dead}, Alive at 3m: n={n_alive}")

# Loop through all variables except the outcome
for var in test_df_death3m_tab1.drop(['bi0.25','bi1','death0.25','death1'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = test_df_death3m_tab1[var].value_counts(dropna=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_3m, alive_3m)
        
        for level in levels:
            # dead group
            count_dead = (dead_3m[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_3m[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            test_df_death3m_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Dead at 3m': f"{count_dead} ({prop_dead:.1f}%)",
                'Alive at 3m': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_3m, alive_3m)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_3m[var].mean()
        dead_sd = dead_3m[var].std()
        dead_median = dead_3m[var].median()
        dead_iqr = dead_3m[var].quantile(0.75) - dead_3m[var].quantile(0.25)

        # alive group
        alive_mean = alive_3m[var].mean()
        alive_sd = alive_3m[var].std()
        alive_median = alive_3m[var].median()
        alive_iqr = alive_3m[var].quantile(0.75) - alive_3m[var].quantile(0.25)

        # Add row for numerical variable
        test_df_death3m_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Dead at 3m': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Alive at 3m': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
test_df_death3m_table1 = pd.DataFrame(test_df_death3m_table1_rows)

# Add sample sizes to column headers
test_df_death3m_table1.columns = [
    'Variable', 
    'Categories', 
    f'Dead at 3m (n={n_dead})', 
    f'Alive at 3m (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 3-month Mortality (TEST SET)")
print("="*80)
print(test_df_death3m_table1.to_string(index=False))

# save to excel
# Subsequent sheets (append mode)
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    test_df_death3m_table1.to_excel(writer, sheet_name = 'death3m_test', index=False)
# The openpyxl engine is required for append mode

### Death 1y

In [ ]:
# =====================================
# OVERALL
# =====================================

# table 1
df_death1y_tab1 = slsr13.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
df_death1y_table1_rows = []

cat_vars = [var for var in df_death1y_tab1.columns if df_death1y_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_1y = df_death1y_tab1[df_death1y_tab1['death1'] == 1]
alive_1y = df_death1y_tab1[df_death1y_tab1['death1'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_1y)
n_alive = len(alive_1y)
n_total = len(df_death1y_tab1)

print(f"Dead at 1y: n={n_dead}, Alive at 1y: n={n_alive}, Total: n={n_total}")

# Loop through all variables except the outcome
for var in df_death1y_tab1.drop(['bi0.25','bi1','death0.25','death1','strk_y'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = df_death1y_tab1[var].value_counts(dropna=False, sort=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_1y, alive_1y)
        
        for level in levels:
            # dead group
            count_dead = (dead_1y[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_1y[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            df_death1y_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Dead at 1y': f"{count_dead} ({prop_dead:.1f}%)",
                'Alive at 1y': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_1y, alive_1y)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_1y[var].mean()
        dead_sd = dead_1y[var].std()
        dead_median = dead_1y[var].median()
        dead_iqr = dead_1y[var].quantile(0.75) - dead_1y[var].quantile(0.25)

        # alive group
        alive_mean = alive_1y[var].mean()
        alive_sd = alive_1y[var].std()
        alive_median = alive_1y[var].median()
        alive_iqr = alive_1y[var].quantile(0.75) - alive_1y[var].quantile(0.25)

        # Add row for numerical variable
        df_death1y_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Dead at 1y': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Alive at 1y': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
df_death1y_table1 = pd.DataFrame(df_death1y_table1_rows)

# Add sample sizes to column headers
df_death1y_table1.columns = [
    'Variable', 
    'Categories', 
    f'Dead at 1y (n={n_dead})', 
    f'Alive at 1y (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 1-year Mortality (OVERALL)")
print("="*80)
print(df_death1y_table1.to_string(index=False))

# save to excel
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    df_death1y_table1.to_excel(writer, sheet_name = 'death1y_overall', index=False)

In [ ]:
# =====================================
# TRAIN DATA
# =====================================

# table 1
train_df_death1y_tab1 = train_df.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
train_df_death1y_table1_rows = []

cat_vars = [var for var in train_df_death1y_tab1.columns if train_df_death1y_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_1y = train_df_death1y_tab1[train_df_death1y_tab1['death1'] == 1]
alive_1y = train_df_death1y_tab1[train_df_death1y_tab1['death1'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_1y)
n_alive = len(alive_1y)
n_total = len(train_df_death1y_tab1)

print(f"Dead at 1y: n={n_dead}, Alive at 1y: n={n_alive}")

# Loop through all variables except the outcome
for var in train_df_death1y_tab1.drop(['bi0.25','bi1','death0.25','death1'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = train_df_death1y_tab1[var].value_counts(dropna=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_1y, alive_1y)
        
        for level in levels:
            # dead group
            count_dead = (dead_1y[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_1y[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            train_df_death1y_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Dead at 1y': f"{count_dead} ({prop_dead:.1f}%)",
                'Alive at 1y': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_1y, alive_1y)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_1y[var].mean()
        dead_sd = dead_1y[var].std()
        dead_median = dead_1y[var].median()
        dead_iqr = dead_1y[var].quantile(0.75) - dead_1y[var].quantile(0.25)

        # alive group
        alive_mean = alive_1y[var].mean()
        alive_sd = alive_1y[var].std()
        alive_median = alive_1y[var].median()
        alive_iqr = alive_1y[var].quantile(0.75) - alive_1y[var].quantile(0.25)

        # Add row for numerical variable
        train_df_death1y_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Dead at 1y': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Alive at 1y': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
train_df_death1y_table1 = pd.DataFrame(train_df_death1y_table1_rows)

# Add sample sizes to column headers
train_df_death1y_table1.columns = [
    'Variable', 
    'Categories', 
    f'Dead at 1y (n={n_dead})', 
    f'Alive at 1y (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 1-year Mortality (TRAINING SET)")
print("="*80)
print(train_df_death1y_table1.to_string(index=False))

# save to excel
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    train_df_death1y_table1.to_excel(writer, sheet_name = 'death1y_train', index=False)

In [ ]:
# =====================================
# TEST DATA
# =====================================

# table 1
test_df_death1y_tab1 = test_df.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
test_df_death1y_table1_rows = []

cat_vars = [var for var in test_df_death1y_tab1.columns if test_df_death1y_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_1y = test_df_death1y_tab1[test_df_death1y_tab1['death1'] == 1]
alive_1y = test_df_death1y_tab1[test_df_death1y_tab1['death1'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_1y)
n_alive = len(alive_1y)
n_total = len(test_df_death1y_tab1)

print(f"Dead at 1y: n={n_dead}, Alive at 1y: n={n_alive}")

# Loop through all variables except the outcome
for var in test_df_death1y_tab1.drop(['bi0.25','bi1','death0.25','death1'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = test_df_death1y_tab1[var].value_counts(dropna=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_1y, alive_1y)
        
        for level in levels:
            # dead group
            count_dead = (dead_1y[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_1y[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            test_df_death1y_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Dead at 1y': f"{count_dead} ({prop_dead:.1f}%)",
                'Alive at 1y': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_1y, alive_1y)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_1y[var].mean()
        dead_sd = dead_1y[var].std()
        dead_median = dead_1y[var].median()
        dead_iqr = dead_1y[var].quantile(0.75) - dead_1y[var].quantile(0.25)

        # alive group
        alive_mean = alive_1y[var].mean()
        alive_sd = alive_1y[var].std()
        alive_median = alive_1y[var].median()
        alive_iqr = alive_1y[var].quantile(0.75) - alive_1y[var].quantile(0.25)

        # Add row for numerical variable
        test_df_death1y_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Dead at 1y': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Alive at 1y': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
test_df_death1y_table1 = pd.DataFrame(test_df_death1y_table1_rows)

# Add sample sizes to column headers
test_df_death1y_table1.columns = [
    'Variable', 
    'Categories', 
    f'Dead at 1y (n={n_dead})', 
    f'Alive at 1y (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 1-year Mortality (TEST SET)")
print("="*80)
print(test_df_death1y_table1.to_string(index=False))

# save to excel
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    test_df_death1y_table1.to_excel(writer, sheet_name = 'death1y_test', index=False)

In [ ]:
# how many samples do i get in the training set vs. temporal validation set?

# get strk_y from first version of extracted dataset
sample_test = slsr10.merge(slsr[['id','strk_y']], on='id', how='left')
print(sample_test.info())

# without efi, on full sample
sample_test_training = sample_test[sample_test['strk_y']<=2020]
print(f'Without EFI, n available for training: {len(sample_test_training)}')
sample_test_tempval = sample_test[sample_test['strk_y']>2020]
print(f'Without EFI, n available for temporal validation: {len(sample_test_tempval)}')

# with efi, on subsample
sample_test_efi_only = sample_test[sample_test['efi_score'].notna()]
print(f'n with non-missing EFI: {len(sample_test_efi_only)}')
sample_test_training_efi = sample_test_efi_only[sample_test_efi_only['strk_y']<=2020]
print(f'Without EFI, n available for training: {len(sample_test_training_efi)}')
sample_test_tempval_efi = sample_test_efi_only[sample_test_efi_only['strk_y']>2020]
print(f'Without EFI, n available for temporal validation: {len(sample_test_tempval_efi)}')

#### Disability 3m

In [ ]:
# =====================================
# OVERALL
# =====================================

# table 1
df_disability3m_tab1 = slsr13[slsr13['death0.25']==0]
df_disability3m_tab1 = df_disability3m_tab1[df_disability3m_tab1['bi0.25'].notna()]
df_disability3m_tab1 = df_disability3m_tab1.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
df_disability3m_table1_rows = []

cat_vars = [var for var in df_disability3m_tab1.columns if df_disability3m_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
disability_3m = df_disability3m_tab1[df_disability3m_tab1['bi0.25'] == 1]
nodisability_3m = df_disability3m_tab1[df_disability3m_tab1['bi0.25'] == 0]

# Total sample sizes for percentage calculations
n_disability = len(disability_3m)
n_nodisability = len(nodisability_3m)
n_total = len(df_disability3m_tab1)

print(f"Disabled at 3m: n={n_disability}, Not disabled at 3m: n={n_nodisability}, Total: n={n_total}")

# Loop through all variables except the outcome
for var in df_disability3m_tab1.drop(['bi0.25','bi1','death0.25','death1','ddestin_binary','strk_y'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = df_disability3m_tab1[var].value_counts(dropna=False, sort=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, disability_3m, nodisability_3m)
        
        for level in levels:
            # dead group
            count_dead = (disability_3m[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_disability * 100

            # alive group
            count_alive = (nodisability_3m[var] == level).sum()
            prop_alive = count_alive / n_nodisability * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            df_disability3m_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Disabled at 3m': f"{count_dead} ({prop_dead:.1f}%)",
                'Not disabled at 3m': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, disability_3m, nodisability_3m)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        disability_mean = disability_3m[var].mean()
        disability_sd = disability_3m[var].std()
        disability_median = disability_3m[var].median()
        disability_iqr = disability_3m[var].quantile(0.75) - disability_3m[var].quantile(0.25)

        # alive group
        nodisability_mean = nodisability_3m[var].mean()
        nodisability_sd = nodisability_3m[var].std()
        nodisability_median = nodisability_3m[var].median()
        nodisability_iqr = nodisability_3m[var].quantile(0.75) - nodisability_3m[var].quantile(0.25)

        # Add row for numerical variable
        df_disability3m_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Disabled at 3m': f"{disability_mean:.2f} ({disability_sd:.2f}) / {disability_median:.2f} ({disability_iqr:.2f})",
            'Not disabled at 3m': f"{nodisability_mean:.2f} ({nodisability_sd:.2f}) / {nodisability_median:.2f} ({nodisability_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
df_disability3m_table1 = pd.DataFrame(df_disability3m_table1_rows)

# Add sample sizes to column headers
df_disability3m_table1.columns = [
    'Variable', 
    'Categories', 
    f'Disabled at 3m (n={n_disability})', 
    f'Not disabled at 3m (n={n_nodisability})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 3-month Disability (OVERALL)")
print("="*80)
print(df_disability3m_table1.to_string(index=False))

# save to excel
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    df_disability3m_table1.to_excel(writer, sheet_name = 'disability_3m_overall', index=False)

### Disability 1y

In [ ]:
# =====================================
# OVERALL
# =====================================

# table 1
df_disability1y_tab1 = slsr13[slsr13['death1']==0]
df_disability1y_tab1 = df_disability1y_tab1[df_disability1y_tab1['bi1'].notna()]
df_disability1y_tab1 = df_disability1y_tab1.drop('id',axis=1)

# Create empty list to store results (better than DataFrame)
df_disability1y_table1_rows = []

cat_vars = [var for var in df_disability1y_tab1.columns if df_disability1y_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
disability_1y = df_disability1y_tab1[df_disability1y_tab1['bi1'] == 1]
nodisability_1y = df_disability1y_tab1[df_disability1y_tab1['bi1'] == 0]

# Total sample sizes for percentage calculations
n_disability = len(disability_1y)
n_nodisability = len(nodisability_1y)
n_total = len(df_disability1y_tab1)

print(f"Disabled at 1y: n={n_disability}, Not disabled at 1y: n={n_nodisability}, Total: n={n_total}")

# Loop through all variables except the outcome
for var in df_disability1y_tab1.drop(['bi0.25','bi1','death0.25','death1','ddestin_binary','strk_y'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = df_disability1y_tab1[var].value_counts(dropna=False, sort=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, disability_1y, nodisability_1y)
        
        for level in levels:
            # dead group
            count_dead = (disability_1y[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_disability * 100

            # alive group
            count_alive = (nodisability_1y[var] == level).sum()
            prop_alive = count_alive / n_nodisability * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            df_disability1y_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Disabled at 1y': f"{count_dead} ({prop_dead:.1f}%)",
                'Not disabled at 1y': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, disability_1y, nodisability_1y)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        disability_mean = disability_1y[var].mean()
        disability_sd = disability_1y[var].std()
        disability_median = disability_1y[var].median()
        disability_iqr = disability_1y[var].quantile(0.75) - disability_1y[var].quantile(0.25)

        # alive group
        nodisability_mean = nodisability_1y[var].mean()
        nodisability_sd = nodisability_1y[var].std()
        nodisability_median = nodisability_1y[var].median()
        nodisability_iqr = nodisability_1y[var].quantile(0.75) - nodisability_1y[var].quantile(0.25)

        # Add row for numerical variable
        df_disability1y_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Disabled at 1y': f"{disability_mean:.2f} ({disability_sd:.2f}) / {disability_median:.2f} ({disability_iqr:.2f})",
            'Not disabled at 1y': f"{nodisability_mean:.2f} ({nodisability_sd:.2f}) / {nodisability_median:.2f} ({nodisability_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
df_disability1y_table1 = pd.DataFrame(df_disability1y_table1_rows)

# Add sample sizes to column headers
df_disability1y_table1.columns = [
    'Variable', 
    'Categories', 
    f'Disabled at 1y (n={n_disability})', 
    f'Not disabled at 1y (n={n_nodisability})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by 1-year Disability (OVERALL)")
print("="*80)
print(df_disability1y_table1.to_string(index=False))

# save to excel
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    df_disability1y_table1.to_excel(writer, sheet_name = 'disability_1y_overall', index=False)

### Discharge Destination

In [ ]:
# =====================================
# OVERALL
# =====================================

# table 1
df_death3m_tab1 = slsr13.drop('id',axis=1)
df_death3m_tab1 = df_death3m_tab1[df_death3m_tab1['ddestin_binary'].notna()]

# Create empty list to store results (better than DataFrame)
df_death3m_table1_rows = []

cat_vars = [var for var in df_death3m_tab1.columns if df_death3m_tab1[var].dtype=='object']
print(f'Categorical variables: {cat_vars}')

# Split data by outcome
dead_3m = df_death3m_tab1[df_death3m_tab1['ddestin_binary'] == 1]
alive_3m = df_death3m_tab1[df_death3m_tab1['ddestin_binary'] == 0]

# Total sample sizes for percentage calculations
n_dead = len(dead_3m)
n_alive = len(alive_3m)
n_total = len(df_death3m_tab1)

print(f"Discharged to institution: n={n_dead}, Discharged to home care: n={n_alive}, Total: n={n_total}")

# Loop through all variables except the outcome
for var in df_death3m_tab1.drop(['bi0.25','bi1','death0.25','death1','ddestin_binary','strk_y'],axis=1).columns:
    
    # categorical variables
    if var in cat_vars:
        levels = df_death3m_tab1[var].value_counts(dropna=False, sort=False).index.tolist()

        # Calculate p-value once for the whole variable
        p_value, test_used = calc_cat_pvalue(var, dead_3m, alive_3m)
        
        for level in levels:
            # dead group
            count_dead = (dead_3m[var] == level).sum() # use .sum() for Boolean series
            prop_dead = count_dead / n_dead * 100

            # alive group
            count_alive = (alive_3m[var] == level).sum()
            prop_alive = count_alive / n_alive * 100

            # Add row (only show p-value on first level of each variable)
            if level == levels[0]:
                p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"
                test_display = f" ({test_used})"
            else:
                p_value_display = ""
                test_display = ""

            df_death3m_table1_rows.append({
                'Variable': var if level == levels[0] else '',  # Only show variable name on first row
                'Categories': str(level),
                'Discharged to institution': f"{count_dead} ({prop_dead:.1f}%)",
                'Discharged to home care': f"{count_alive} ({prop_alive:.1f}%)",
                'P-value': p_value_display + test_display
            })

    # if numerical variable
    else:
        # Calculate p-value
        p_value, test_used = calc_num_pvalue(var, dead_3m, alive_3m)
        # Format p-value
        p_value_display = f"{p_value:.3f}" if not pd.isna(p_value) else "NA"

        # dead group
        dead_mean = dead_3m[var].mean()
        dead_sd = dead_3m[var].std()
        dead_median = dead_3m[var].median()
        dead_iqr = dead_3m[var].quantile(0.75) - dead_3m[var].quantile(0.25)

        # alive group
        alive_mean = alive_3m[var].mean()
        alive_sd = alive_3m[var].std()
        alive_median = alive_3m[var].median()
        alive_iqr = alive_3m[var].quantile(0.75) - alive_3m[var].quantile(0.25)

        # Add row for numerical variable
        df_death3m_table1_rows.append({
            'Variable': var,
            'Categories': 'Mean (SD) / Median (IQR)',
            'Discharged to institution': f"{dead_mean:.2f} ({dead_sd:.2f}) / {dead_median:.2f} ({dead_iqr:.2f})",
            'Discharged to home care': f"{alive_mean:.2f} ({alive_sd:.2f}) / {alive_median:.2f} ({alive_iqr:.2f})",
            'P-value': f"{p_value_display} ({test_used})"
        })
    
# Convert list of rows to DataFrame
df_death3m_table1 = pd.DataFrame(df_death3m_table1_rows)

# Add sample sizes to column headers
df_death3m_table1.columns = [
    'Variable', 
    'Categories', 
    f'Discharged to institution (n={n_dead})', 
    f'Discharged to home care (n={n_alive})', 
    'P-value'
]

# Display the table
print("\n" + "="*80)
print("TABLE 1: Patient Characteristics by Discharge destination (OVERALL)")
print("="*80)
print(df_death3m_table1.to_string(index=False))

# save to excel
with pd.ExcelWriter(onedrive_folder+results_folder+table1_file, engine = 'openpyxl', mode = 'a', if_sheet_exists = 'replace') as writer:
    df_death3m_table1.to_excel(writer, sheet_name = 'ddestin_overall', index=False)